# Phishing URL Detection
## Importing Data 

In [77]:
# https://pandas.pydata.org/docs/reference/index.html
import pandas as pd

#loads the CSV file and stores it as "df" or Data Frame
df = pd.read_csv("dataset/malicious_phish.csv")
#gives us the size of the table
print(df.shape)
#tells us what data types are in each column
print(df.dtypes)
#counts the missing values if there are any
print(df.isna().sum())
#counts the amount of each value appear in the data
print(df["type"].value_counts())
#shows 10 random rows as an example
df.sample(10)

(651191, 2)
url     object
type    object
dtype: object
url     0
type    0
dtype: int64
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


,url,type
439784,http://sfglobe.com/2015/01/25/saturn-sits-wher...,benign
360920,hotelonthelake.com/,benign
318823,thevisagateway.com,phishing
278996,http://himado.in/?sort=today_view_cnt&cat=%E3%...,benign
293916,moneymakergroup.com/BrantonsGroup-brantons-t37...,benign
267069,fandango.com/melissagalianos/filmography/p265509,benign
282456,amazon.com/Kodak-Mini-Video-Camera-Card/dp/B00...,benign
208603,allbusiness.com/companyindex/Missouri/Kansas_C...,benign
396610,betinf.com/italy_transfer.htm,benign
85726,last.fm/music/Skatterman+%26+Snug+Brim,benign


## Cleaning Data

In [78]:
#uses "dropna" to get rid of any missing entrys
#https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html#
df = df.dropna(subset=["url", "type"])
#turns all values in the URL colomn into strings and then removes spaces, tabs and other stuff so that it make it easier to read
df["url"] = df["url"].astype(str).str.strip()
#creates a column of bool values where True is a URL that has text and False are just blank
df = df[df["url"] != ""]
df.sample(10)

,url,type
551939,http://31.146.124.122:36807/Mozi.m,malware
637761,groups.yahoo.com/group/VTNT/,phishing
323545,masterliness.com/a/List.of.postal.codes.in.Can...,benign
241719,marketwire.com/press-release/JAG-Mines-Ltd-Oli...,benign
649010,www.freewebs.com/worldslargestdungeon/,phishing
258627,http://www.kidsfestshellharbour.com.au/index.p...,defacement
217471,tuac.org/en/public/tuac/affiliates.phtml,benign
56672,bestplaces.net/county/michigan/kent,benign
456724,thecrowncenter.com/,benign
369775,tvguide.com/celebrities/peter-cullen/169896,benign


In [79]:
#use "map" to assign a int to each type so that it is easier to sort and understand
#https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html
df["type"] = df["type"].map({"phishing": 0, "benign": 1, "defacement": 2, "malware": 3})
#cleans out all the rows that arnt in the dictionary above
df = df.dropna(subset=["type"])
#converts all the types to int incase any werent
df["type"] = df["type"].astype(int)

def simple_preprocess(url):
    if pd.isna(url):
        return ""
    #strips the www. off the urls so its easier to read
    url = url.replace('https://', '').replace('http://', '').replace('www.', '')
    return url.lower().strip()

df['clean_url'] = df['url'].apply(simple_preprocess)

#gives me a test sample to see if the map worked
df.sample(10)

,url,type,clean_url
451483,http://www.cefme.bn.it/index.php?option=com_co...,2,cefme.bn.it/index.php?option=com_content&view=...
106402,rsblawfirm.com/local.html,1,rsblawfirm.com/local.html
277883,mediamanint.com/profiles/jameson.html,1,mediamanint.com/profiles/jameson.html
616497,www.webreference.com/xml/column13/,0,webreference.com/xml/column13/
214477,bigsoccer.com/clubs/n.-america-concacaf/united...,1,bigsoccer.com/clubs/n.-america-concacaf/united...
142528,soldlab.com/content/readMore1/infobox/featured...,1,soldlab.com/content/readmore1/infobox/featured...
237451,chacha.com/question/what-does-bodine-mean,1,chacha.com/question/what-does-bodine-mean
566880,www.picasasicilia.com//components/com_realtyna...,1,picasasicilia.com//components/com_realtyna/js/...
359746,celebgalz.com/shane-michel-shane-michel-hospit...,1,celebgalz.com/shane-michel-shane-michel-hospit...
481223,treedoctor.anr.msu.edu/ash/index.html,1,treedoctor.anr.msu.edu/ash/index.html


## Machine Learning model

In [80]:
#gets the function that will split the data into the training set and the test set
#https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
from sklearn.model_selection import train_test_split
#
from sklearn.feature_extraction.text import TfidfVectorizer
#gets the Random Forest model that make a collection of decision trees that vote of the best answer
#https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html 
from sklearn.ensemble import RandomForestClassifier

In [81]:
#create's a function that creates a dictionary that discribes the URL
def extract_simple_features(url):
    features = {}
    #the length of the URL
    features['length'] = len(url)
    #how many dots are in the URL
    features['num_dots'] = url.count('.')
    #how many slashes are in the URL
    features['num_slashes'] = url.count('/')
    #how many numbers are in the URL
    features['num_digits'] = sum(c.isdigit() for c in url)
    #dose it contain https
    features['has_https'] = 1 if 'https' in url else 0
    return features

features_list = []
#loops over each URL and enters the features into a list
for url in df['clean_url']:
    features_list.append(extract_simple_features(url))
    #tures the list into a Data Frame and attaches it to right side of the original
features_df = pd.DataFrame(features_list)
df = pd.concat([df, features_df], axis=1)
df.sample(10)

,url,type,clean_url,length,num_dots,num_slashes,num_digits,has_https
161144,http://www.enolr.com/event-calendar/day/date/2...,2,enolr.com/event-calendar/day/date/2013-01-10/1...,52,2,5,10,0
292968,http://www.bjorklidensgolfklubb.se/index.php/a...,2,bjorklidensgolfklubb.se/index.php/aktiviteter/...,73,2,5,2,0
428842,en.wikipedia.org/wiki/Soul_Man_(TV_series),1,en.wikipedia.org/wiki/soul_man_(tv_series),42,2,2,0,0
14517,stockhouse.com/Bullboards/SymbolList.aspx?s=AC...,1,stockhouse.com/bullboards/symbollist.aspx?s=ac...,56,3,2,0,0
284045,http://mybigfishsushi.com/index.php?option=com...,2,mybigfishsushi.com/index.php?option=com_poll&i...,77,2,1,2,0
353280,http://www.teknosima.it/index.php?option=com_c...,2,teknosima.it/index.php?option=com_content&view...,68,2,1,2,0
64934,amazon.com/Dream-Games-Art-Robert-Schwartz/dp/...,1,amazon.com/dream-games-art-robert-schwartz/dp/...,56,1,3,10,0
92091,imdb.com/Sections/Awards/Academy_of_Science_Fi...,1,imdb.com/sections/awards/academy_of_science_fi...,85,1,4,4,0
77471,http://clickandbuy.elaine-asp.de/go/503147e3/7...,0,clickandbuy.elaine-asp.de/go/503147e3/7ftajglq,46,2,3,8,0
328327,kinston.com/articles/linda-6312-davis-patricia...,1,kinston.com/articles/linda-6312-davis-patricia...,51,2,2,4,0


In [82]:
#setting out the inputs the model will learn from
#X is the clean URLs which will be the input it will learn from
X = df['clean_url']
#Y is the type colume and the answers it will predict 
Y = df['type']
#this splits the data into 80% training data and 20% teasting data
X_train, X_test, Y_train, Y_test = train_test_split(
    #random state will shuffle so that you dont get that same split ever time and stratify will keep the class proportions the same in both sets
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [83]:
#this sets up the vectorizer as the model cant read text so it converts each URL into a matrix of of numbers based on the pieces it contains or TF-IDF (this is the confusing one)
#https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html 
vectorizer = TfidfVectorizer(
    #max features keeps only the 500 most useful parts
    max_features=500,
    #ngarm range expans the range at wich tokens are made, insted of single wrods like linkedin, com, au it looks for linkedin com and com au (its complicated and confusing I know)
    ngram_range=(1, 2)
)
#fit transform will learn the vocabulary and returne a document term matrix
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
#starts the learning model by creating 50 trees
clf = RandomForestClassifier(n_estimators=50, random_state=42)
#learns the link between the vector's and the lables 
clf.fit(X_train_vec, Y_train)

KeyboardInterrupt: 

In [ ]:
#Feeds the test URLs into the trained model and receves its guesses and stores them in Y pred
Y_pred = clf.predict(X_test_vec)
# Shows an avrage accuracy of the model
print("Accuracy:", (Y_pred == Y_test).mean())

IndexError: list index out of range

In [ ]:
def predict_url(url):
    #re-cleans the URL so that the model dosent see diffrently-shaped input and learns from it
    clean_url = simple_preprocess(url)
    #transforms the cleaned URL back into the TFIDF number format
    url_vec = vectorizer.transform([clean_url])
    # gives an array with a single prediction and pulls it out of the array
    prediction = clf.predict(url_vec)[0]
    #returns the models confiednce for every class
    probability = clf.predict_proba(url_vec)[0]
    
    #packages the URL, the type the model predicts and how confident it is together into a dictonary
    return {
        'url': url,
        'prediction': prediction,
        'confidence': max(probability)
    }
#runs through some hand picked URLs and gives the URL, the type the model predicts and how confident it is 
test_urls = [
    "https://www.paypal.com/login",
    "https://clashofclansgemgenerator.webflow.io/", 
    "https://www.twitch.tv/",
    "https://www.google.com",
    "https://github.com/microsoft/vscode",
    "http://amazon-security-alert.ml",
    "http://paypal.verify-account.com",
    "http://google.com",
    "http://free-download-crack.tk",
    "https://hianime.ro/home/",
    "https://calendar.google.com",
    "https://discord.com/",
    "https://store.steampowered.com/"
    "https://cornhub.website/"
    "https://en.wikipedia.org/wiki/Emu_War"
]
#prints out the package in a easily read format for the user
for url in test_urls:
    result = predict_url(url)
    print(f"URL: {url}")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.2f}\n")

# Visualisation
mostly a work in progress

In [ ]:
from sklearn import metrics
import matplotlib.pyplot as plt
import numpy

actual = Y_test
predicted = Y_pred

confusion_matrix = metrics.confusion_matrix(actual, predicted)

cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])

cm_display.plot()
plt.show()